<a href="https://colab.research.google.com/github/nithinchowdary59/dataviz-exercises-Nithin_Chowdary_Raya/blob/main/Quiz/dataviz_quiz.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data Visualization — Quiz (Lectures 1–7)

**Time:** 90 minutes
**Tools:** Plotly only (`plotly.express` first, then `update_traces` / `update_layout` / `update_xaxes` / `update_yaxes`)

**Your task:** answer each analytical question with **one polished, explanatory visualization**. For every question **you decide which chart type fits best** — and justify the choice in one or two sentences.

**Every figure must have:**
1. An **insight title** that states the finding, not just the variables
2. `template='plotly_white'` and a decluttered layout (no unnecessary gridlines, legends or axis titles)
3. A **CVD-safe palette** — no red/green; grey for context + one accent colour where it helps
4. Colour that **encodes meaning**, not decoration
5. Direct labels or annotations where they help the reader

**Chart types covered in the course** — choose from these:
line chart · sorted bar chart · stacked / 100% stacked bar chart · scatter plot · bubble chart · histogram · box plot · violin plot · choropleth map · treemap

> 💡 Each question has one chart type that fits it best, and the questions together cover most of the list above. If two of your answers use the same chart type, check the question again.


**Submit:** this notebook pushed to your personal repo as `quiz/dataviz_quiz.ipynb`

## The dataset — World Population 1970–2022

**Source:** [World Population Dataset](https://www.kaggle.com/datasets/iamsouravbanerjee/world-population-dataset) on Kaggle, compiled from Worldometer (UN World Population Prospects estimates). **234 rows × 16 columns — one row per country or territory, no missing values.**

| Column | Type | Description |
|---|---|---|
| `Country`, `Capital` | categorical | Country / territory and its capital |
| `Continent` | categorical | Africa, Asia, Europe, North America, Oceania, South America |
| `CCA3` | spatial | ISO-3 country code — use with `locationmode='ISO-3'` |
| `1970 Population` … `2022 Population` | temporal | Population in 1970, 1980, 1990, 2000, 2010, 2015, 2020, 2022 |
| `Area` | numerical | Land area in km² |
| `Density` | numerical | People per km² (2022) |
| `World share (%)` | numerical | Share of world population in 2022 |
| `Rank` | numerical | Rank by 2022 population |

In [6]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


**The next setup cell also gives you `df_long` — the same data in **tidy (long) format** with one row per country per year. Use `df` for one-year questions and `df_long` for questions about change over time**

In [7]:
import os

for root, dirs, files in os.walk('/content/drive/MyDrive'):
    if 'world_population.csv' in files:
        print(os.path.join(root, 'world_population.csv'))
        break

/content/drive/MyDrive/world_population.csv


In [9]:
import pandas as pd
import plotly.express as px
import numpy as np

df = pd.read_csv('/content/drive/MyDrive/world_population.csv')

# Shorter column names
df = df.rename(columns={'Country/Territory': 'Country', 'Area (km²)': 'Area',
                        'Density (per km²)': 'Density', 'World Population Percentage': 'World share (%)'})

# Tidy (long) version: one row per country per year
year_cols = [c for c in df.columns if c.endswith('Population')]
df_long = df.melt(id_vars=['CCA3', 'Country', 'Continent'], value_vars=year_cols,
                  var_name='Year', value_name='Population')

df_long['Year'] = df_long['Year'].str[:4].astype(int)     # '2022 Population' → 2022

print(df.shape, df_long.shape)


(234, 16) (1872, 5)


---
## Q1 — Which continent is growing fastest?

How has the population of each continent **grown since 1970, relative to its 1970 size** — and which continent is growing fastest?


**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

A line chart is suitable because it shows how population growth changes over time. I use an index where 1970 = 100 for every continent, allowing us to compare relative growth rather than total population.

In [10]:
print(df.shape)
print(df.head())
print(df.columns.tolist())

(234, 16)
   Rank CCA3         Country           Capital Continent  2022 Population  \
0    36  AFG     Afghanistan             Kabul      Asia         41128771   
1   138  ALB         Albania            Tirana    Europe          2842321   
2    34  DZA         Algeria           Algiers    Africa         44903225   
3   213  ASM  American Samoa         Pago Pago   Oceania            44273   
4   203  AND         Andorra  Andorra la Vella    Europe            79824   

   2020 Population  2015 Population  2010 Population  2000 Population  \
0         38972230         33753499         28189672         19542982   
1          2866849          2882481          2913399          3182021   
2         43451666         39543154         35856344         30774621   
3            46189            51368            54849            58230   
4            77700            71746            71519            66097   

   1990 Population  1980 Population  1970 Population     Area   Density  \
0         106

In [11]:
# Q1
# Convert population data into long format
year_cols = [col for col in df.columns if col.endswith('Population')]

df_long = df.melt(
    id_vars=['Country', 'Continent'],
    value_vars=year_cols,
    var_name='Year',
    value_name='Population'
)

df_long['Year'] = df_long['Year'].str[:4].astype(int)

# Calculate total population by continent and year
continent_pop = (
    df_long.groupby(['Continent', 'Year'])['Population']
    .sum()
    .reset_index()
)

# Calculate relative growth (1970 = 100)
continent_pop['Growth Index'] = (
    continent_pop.groupby('Continent')['Population']
    .transform(lambda x: x / x.iloc[0] * 100)
)

latest = continent_pop[continent_pop['Year'] == 2022]
fastest = latest.loc[latest['Growth Index'].idxmax(), 'Continent']

# Create line chart
fig = px.line(
    continent_pop,
    x='Year',
    y='Growth Index',
    color='Continent',
    markers=True,
    title=f'{fastest} Has the Fastest Relative Population Growth Since 1970',
    template='plotly_white'
)

fig.update_layout(
    title_x=0.5,
    xaxis_title='Year',
    yaxis_title='Population Growth Index (1970 = 100)',
    yaxis_showgrid=False
)

fig.show()

---
## Q2 — Which countries added the most people?

Which **15 countries added the most people** between 1970 and 2022 — and how many of them are in **Africa**?

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

I chose a horizontal bar chart because it makes it easy to compare population increases across the top 15 countries. The countries are sorted by population increase, from highest to lowest. I used blue to highlight African countries and grey for the remaining countries so that the African countries are easy to identify.



In [13]:
# Q2

# Calculate population increase between 1970 and 2022
df['People Added'] = df['2022 Population'] - df['1970 Population']

# Select the top 15 countries
top15 = df.nlargest(15, 'People Added').copy()

# Highlight African countries
top15['Region'] = top15['Continent'].apply(
    lambda x: 'Africa' if x == 'Africa' else 'Other continents'
)

# Count African countries in the top 15
africa_count = (top15['Continent'] == 'Africa').sum()

print("Number of African countries in the top 15:", africa_count)

# Sort for horizontal bar chart
top15 = top15.sort_values('People Added', ascending=True)

# Create chart
fig = px.bar(
    top15,
    x='People Added',
    y='Country',
    orientation='h',
    color='Region',
    text='People Added',
    color_discrete_map={
        'Africa': '#2878B5',
        'Other continents': '#B8B8B8'
    },
    title=f'Top 15 Countries by Population Increase (1970–2022) — {africa_count} from Africa',
    template='plotly_white'
)

fig.update_traces(
    texttemplate='%{text:,.0f}',
    textposition='outside'
)

fig.update_layout(
    title_x=0.5,
    xaxis_title='People Added',
    yaxis_title='',
    xaxis_showgrid=False,
    legend_title=''
)
fig.update_layout(
    margin=dict(l=100, r=130, t=80, b=60)
)

fig.show()


Number of African countries in the top 15: 4


---
## Q3 — Do bigger countries have more people?

Is there a relationship between a country's **land area** and its **2022 population** — does it differ by **continent**, and which countries clearly break the pattern?

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

I chose a scatter plot because it shows the relationship between a country's land area and its population. I used different colours for each continent to compare regional patterns. Logarithmic scales help display countries with very different land areas and populations. I also added country labels for some extreme values to identify countries that break the general pattern.

I will use:
- X-axis: Land area
- Y-axis: 2022 population
- Colour: Continent
- Logarithmic scales: To make comparisons easier
- Labels: Three largest countries by area and three largest by population

In [14]:
# Q3

top_area = df.nlargest(3, 'Area')['Country']
top_population = df.nlargest(3, '2022 Population')['Country']


df['Label'] = df['Country'].where(
    df['Country'].isin(top_area) |
    df['Country'].isin(top_population),
    ''
)

fig = px.scatter(
    df,
    x='Area',
    y='2022 Population',
    color='Continent',
    hover_name='Country',
    text='Label',
    log_x=True,
    log_y=True,
    title='Land Area and Population Show Different Patterns Across Countries',
    template='plotly_white',
    opacity=0.75
)

fig.update_traces(
    textposition='top center',
    marker=dict(size=9)
)

fig.update_layout(
    title_x=0.5,
    xaxis_title='Land Area (km², logarithmic scale)',
    yaxis_title='2022 Population (logarithmic scale)',
    xaxis_showgrid=False,
    yaxis_showgrid=False
)

fig.show()


---
## Q4 — How crowded are the continents?

How does **population density vary within and across continents**? Show the spread, not just the average.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

I chose a box plot because it shows how population density varies within each continent. It displays the median, spread and outliers, making it easier to compare continents. I used different colours for each continent and logarithmic scaling to make countries with very different population densities easier to compare.


The box plot shows that population density varies considerably within and across continents. Asia and Europe have relatively high median densities, while South America has a lower median. Several countries appear as outliers, indicating unusually high population density compared with other countries in their continents. The logarithmic scale helps compare countries with very different density values.

In [16]:
#Q4


fig = px.box(
    df,
    x='Continent',
    y='Density',
    color='Continent',
    points='outliers',
    log_y=True,
    title='Population Density Varies Widely Within and Across Continents',
    template='plotly_white'
)

fig.update_layout(
    title_x=0.5,
    xaxis_title='Continent',
    yaxis_title='Population Density (people per km², log scale)',
    showlegend=False,
    yaxis_showgrid=False
)

fig.show()


---
## Q5 — Where did populations grow — and where did they shrink?

Which countries' populations **grew and which shrank between 2000 and 2022** (percentage change), and where in the world are they?

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

I chose a choropleth map because it shows population percentage change across countries geographically. Countries with population growth and decline are represented using different colours. A diverging colour scale centred at zero makes it easy to distinguish growing countries from shrinking countries. Hover information provides the country name and percentage change.

In [ ]:
# Q5
# YOUR CODE HERE

---
## Q6 — Where do the world's 8 billion people live?

How is the **2022 world population** distributed across **continents and the countries within them**? Show both levels in one figure.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE*

In [17]:
# Q6

# Calculate population percentage change (2000–2022)
df['Population Change (%)'] = (
    (df['2022 Population'] - df['2000 Population'])
    / df['2000 Population']
) * 100

# Identify growth and decline
df['Growth Status'] = df['Population Change (%)'].apply(
    lambda x: 'Growth' if x > 0 else 'Decline' if x < 0 else 'No change'
)

# Count countries in each category
print("Population growth and decline:")
print(df['Growth Status'].value_counts())

# Create choropleth map
fig = px.choropleth(
    df,
    locations='CCA3',
    color='Population Change (%)',
    hover_name='Country',
    hover_data={
        'Population Change (%)': ':.2f',
        'Growth Status': True,
        'CCA3': False
    },
    color_continuous_scale='PuOr',
    color_continuous_midpoint=0,
    title='Population Growth and Decline Across Countries (2000–2022)',
    template='plotly_white'
)

fig.update_layout(
    title_x=0.5,
    geo=dict(
        showframe=False,
        showcoastlines=True,
        projection_type='natural earth'
    ),
    coloraxis_colorbar_title='Population Change (%)'
)

fig.show()


Population growth and decline:
Growth Status
Growth     200
Decline     34
Name: count, dtype: int64


---
## Q7 — Whose share of the world is rising?

How have the shares of world population held by **Africa, Asia and Europe** changed from 1970 to 2022? Group the other three continents as `'Rest of the world'`.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

I chose a multi-line chart because it clearly shows how the share of world population changes over time. Separate lines represent Africa, Asia, Europe and the Rest of the world. This makes it easier to compare increasing and decreasing trends across the four groups.

In [19]:

fig = px.line(
    grouped,
    x='Year',
    y='World Share (%)',
    color='Continent Group',
    markers=True,
    category_orders={
        'Continent Group': [
            'Africa', 'Asia', 'Europe', 'Rest of the world'
        ]
    },
    color_discrete_map={
        'Africa': '#2878B5',
        'Asia': '#E69F00',
        'Europe': '#009E73',
        'Rest of the world': '#888888'
    },
    title='How the World Population Share Changed Across Continents (1970–2022)',
    template='plotly_white'
)

fig.update_layout(
    title_x=0.5,
    xaxis_title='Year',
    yaxis_title='Share of World Population (%)',
    xaxis_showgrid=False,
    yaxis_showgrid=False,
    legend_title=''
)

fig.show()
